<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Topic_top_author_sentimen_VER2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Install Dependency

In [19]:
import os, sys, shutil
from google.colab import userdata

REPO_DIR = "/content/GREAT-Tools"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q https://{userdata.get('GH_TOKEN')}@github.com/azmkto/GREAT-Tools.git {REPO_DIR}

!pip uninstall -y torch torchvision torchaudio
!pip install -q torch torchvision torchaudio
!pip install -q -U pandas tqdm emoji openpyxl ftfy gliner transformers

print("[PENDING] Restart runtime sekarang: Runtime -> Restart session. Baru jalankan cell berikutnya.")

Found existing installation: torch 2.14.0
Uninstalling torch-2.14.0:
  Successfully uninstalled torch-2.14.0
Found existing installation: torchvision 0.29.0
Uninstalling torchvision-0.29.0:
  Successfully uninstalled torchvision-0.29.0
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
[PENDING] Restart runtime sekarang: Runtime -> Restart session. Baru jalankan cell berikutnya.


## 1. Import Library

In [20]:
import json
import os
import random
import re
import sys
import time
from collections import Counter
from datetime import date

import emoji
import pandas as pd
from google.colab import ai, drive
from IPython.display import display
from tqdm.auto import tqdm

# restart runtime menghapus sys.path & variabel, jadi repo didaftarkan ulang di sini
REPO_DIR = "/content/GREAT-Tools"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from great.labels import is_media_account
from great.schema import EXPORT_COLUMNS
from great.text import clean_for_bert, clean_for_ner

tqdm.pandas()

print("great loaded from:", sys.modules["great"].__file__)
print("Commit terakhir  :", os.popen(f"git -C {REPO_DIR} rev-parse --short HEAD").read().strip())
print("Model Colab AI   :", ai.list_models())

great loaded from: /content/GREAT-Tools/great/__init__.py
Commit terakhir  : 8fe67ab
Model Colab AI   : ['google/gemini-2.5-flash', 'google/gemini-2.5-flash-lite', 'google/gemini-2.5-pro', 'google/gemini-3.1-pro-preview', 'google/gemini-3.5-flash']


# 2. Colab AI

In [21]:
AI_MODEL = "google/gemini-2.5-flash"   # pilih dari ai.list_models()
SLEEP_BETWEEN_CALLS = 4
api_call_count = 0


def is_quota_error(e):
    msg = str(e).lower()
    return "429" in msg or "resource_exhausted" in msg or "quota" in msg


def parse_json_text(text):
    """JSON pertama di jawaban model (Colab AI tidak punya mode JSON)."""
    starts = [i for i in (text.find("["), text.find("{")) if i >= 0]
    if not starts:
        raise ValueError("tidak ada JSON di jawaban model")
    return json.JSONDecoder().raw_decode(text[min(starts):])[0]


def ask_json(prompt, parse=lambda x: x, retry=4):
    """Colab AI -> JSON -> parse(). Kuota habis langsung di-raise; None kalau semua retry gagal."""
    global api_call_count
    for attempt in range(retry):
        try:
            api_call_count += 1
            return parse(parse_json_text(ai.generate_text(prompt, model_name=AI_MODEL)))
        except Exception as e:
            if is_quota_error(e):
                raise
            wait = (20 if "503" in str(e) or "UNAVAILABLE" in str(e) else 5) * (attempt + 1)
            print(f"[WARN] AI gagal ({attempt + 1}/{retry}): {e} -> tunggu {wait}s")
            time.sleep(wait)
    return None


def order_by_index(parsed, n):
    """List objek ber-field 'index' (1..n) -> list urut. Error kalau ada index yang hilang."""
    by_idx = {int(it["index"]): it for it in parsed if isinstance(it, dict) and "index" in it}
    if set(by_idx) != set(range(1, n + 1)):
        raise ValueError(f"Index hasil tidak lengkap: dapat {sorted(by_idx)}, harus 1..{n}")
    return [by_idx[i] for i in range(1, n + 1)]

# 3. TOPIK

In [22]:
PROJECT = "prabowo_vladivostok"
TOPIC = "Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin"
SOURCES = [
    "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true",
]

In [23]:
# # Tema: key = nama tema, value = deskripsi untuk model. "lainnya" wajib ada.
# THEME_DESCRIPTIONS = {
#     "kunjungan_prabowo": (
#         "kunjungan Prabowo ke Rusia, kunjungan kenegaraan Prabowo, lawatan Prabowo, agenda Prabowo di Rusia, "
#         "delegasi Indonesia ke Rusia, Prabowo temui Putin, pertemuan bilateral Prabowo Putin, "
#         "Prabowo di Vladivostok, Prabowo di Moskow, Prabowo hadiri Eastern Economic Forum, "
#         "Prabowo EEF, Prabowo forum ekonomi Rusia, kunjungan presiden RI ke Rusia, "
#         "kerja sama Indonesia Rusia, nota kesepahaman Indonesia Rusia, MoU Indonesia Rusia, "
#         "investasi Rusia ke Indonesia, kunjungan balasan Prabowo, protokoler kunjungan Prabowo, "
#         "rombongan menteri dampingi Prabowo, jadwal kunjungan Prabowo Rusia, "
#         "hasil pertemuan Prabowo Putin, pernyataan bersama Indonesia Rusia, "
#         "kunjungan kenegaraan ke Kremlin, Prabowo di Kremlin, sambutan kenegaraan Prabowo Rusia, "
#         "kereta kepresidenan Rusia, upacara penyambutan Prabowo, kunjungan luar negeri Prabowo, "
#         "diplomasi Prabowo Rusia, agenda strategis Indonesia Rusia, "
#         "lawatan kenegaraan Presiden Prabowo, kunjungan resmi Presiden Prabowo ke Rusia, "
#         "Prabowo Subianto ke Rusia, Prabowo terbang ke Rusia, keberangkatan Prabowo ke Rusia, "
#         "Prabowo tiba di Rusia, Prabowo pulang dari Rusia, kepulangan Prabowo dari Rusia, "
#         "menteri luar negeri dampingi Prabowo, Sugiono dampingi Prabowo, Menlu RI ke Rusia, "
#         "menteri pertahanan dampingi Prabowo, delegasi bisnis Indonesia Rusia, "
#         "pengusaha Indonesia ikut kunjungan Rusia, pebisnis dampingi Prabowo, "
#         "kesepakatan dagang Indonesia Rusia, perjanjian kerja sama Indonesia Rusia, "
#         "kontrak dagang Indonesia Rusia, ekspor impor Indonesia Rusia, "
#         "kerja sama pertahanan Indonesia Rusia, alutsista Rusia untuk Indonesia, "
#         "kerja sama energi Indonesia Rusia, kerja sama nuklir Indonesia Rusia, "
#         "PLTN Rusia Indonesia, Rosatom Indonesia, kerja sama pangan Indonesia Rusia, "
#         "kunjungan Prabowo pasca kunjungan ke China, kunjungan Prabowo pasca KTT, "
#         "reaksi publik kunjungan Prabowo Rusia, kritik kunjungan Prabowo ke Rusia, "
#         "pujian kunjungan Prabowo ke Rusia, kontroversi kunjungan Prabowo Rusia, "
#         "netralitas Indonesia kunjungan Rusia, politik luar negeri bebas aktif Prabowo, "
#         "sikap Barat soal kunjungan Prabowo Rusia, respons AS soal kunjungan Prabowo Rusia, "
#         "istana kepresidenan soal kunjungan Rusia, juru bicara presiden soal kunjungan Rusia, "
#         "foto kunjungan Prabowo Rusia, video kunjungan Prabowo Rusia, momen Prabowo di Rusia, "
#         "red carpet Prabowo Rusia, karpet merah Prabowo Rusia, penyambutan militer Prabowo Rusia"
#     ),
#     "rusia_putin": (
#         "Rusia, Vladimir Putin, Presiden Rusia, Kremlin, Vladivostok, Moskow, Rusia Timur Jauh, "
#         "Eastern Economic Forum, EEF Vladivostok, forum ekonomi Rusia, kebijakan luar negeri Rusia, "
#         "hubungan Rusia dengan negara lain, sanksi terhadap Rusia, sanksi Barat ke Rusia, "
#         "ekonomi Rusia, perdagangan Rusia, energi Rusia, gas Rusia, minyak Rusia, "
#         "militer Rusia, angkatan bersenjata Rusia, perang Rusia Ukraina, konflik Rusia Ukraina, "
#         "geopolitik Rusia, pernyataan Putin, pidato Putin, kebijakan Putin, "
#         "Kementerian Luar Negeri Rusia, duta besar Rusia, kedutaan Rusia, "
#         "kerja sama BRICS, Rusia BRICS, aliansi Rusia, mitra strategis Rusia, "
#         "wilayah Timur Jauh Rusia, pelabuhan Vladivostok, industri Rusia, "
#         "hubungan diplomatik dengan Rusia, kunjungan pejabat asing ke Rusia, "
#         "Kremlin Moskow, Lapangan Merah, Istana Kremlin, juru bicara Kremlin, Dmitry Peskov, "
#         "Sergey Lavrov, Menlu Rusia, diplomasi Rusia, Rusia dan negara Asia, Rusia dan ASEAN, "
#         "Rusia dan Asia Tenggara, kunjungan kepala negara ke Rusia, tamu negara Rusia, "
#         "ekonomi Rusia pasca sanksi, dampak sanksi terhadap Rusia, Rusia dan China, "
#         "Rusia dan India, Rusia di panggung internasional, isolasi Rusia, Rusia G20"
#     ),
#     "lainnya": "topik di luar kategori di atas",
# }

# # Baris tanpa satu pun kata ini langsung "lainnya" (hemat AI + GLiNER).
# # Kosongkan list = semua baris ke AI.
# KW_RELEVAN = [
#     "prabowo", "wowo", "psubianto", "presiden ri", "presiden indonesia",
#     "rusia", "russia", "putin", "kremlin", "moskow", "moscow", "vladivostok",
#     "eastern economic forum", "eef", "rosatom", "lavrov", "peskov", "brics", "ukraina",
# ]

# # Nama lain -> nama baku (untuk laporan)
# ENTITY_ALIASES = {
#     "wowo": "Prabowo", "pak wowo": "Prabowo", "pak prabowo": "Prabowo",
#     "prabowo subianto": "Prabowo", "psubianto": "Prabowo",
#     "vlad putin": "Vladimir Putin", "putin": "Vladimir Putin", "vladimir putin": "Vladimir Putin",
#     "vladivostock": "Vladivostok", "fefu": "Far Eastern Federal University",
# }

# # Akun yang dianggap media khusus di topik ini (tambahan dari daftar di library)
# EXTRA_MEDIA_ACCOUNTS = {"kremlin", "kremlinru"}

# 4. Konfigurasi

Edit bagian ini kalau nama kolom, daftar tema, atau bobot ranking berubah.
Semua cell di bawah memakai variabel dari sini.


In [24]:
drive.mount('/content/drive')

COL_HEADLINE = "Headline"
COL_MENTIONS = "Mentions"
COL_MEDIA = "Media"
COL_AUTHOR_ID = "Author"
COL_FOLLOWERS = "Followers"
COL_RETWEETED = "Retweeted"
COL_FAVOURITED = "Favourited"
EXPECTED_COLS = [c for c in EXPORT_COLUMNS if c != "Location"]   # Location tidak dipakai di sini

GLINER_MODEL_NAME = "urchade/gliner_medium-v2.1"
GLINER_LABELS = ["lokasi", "instansi", "tokoh"]
GLINER_THRESHOLD = 0.5
# Label map IndoBERT tidak di-hardcode, diambil dari config model di Step 6A.
INDOBERT_MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"

# Bobot skor ranking top author (jumlahnya 1.0)
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15

BATCH_SIZE_CLASSIFY = 40
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15

RANKING_MIN_CONFIDENCE = 0.6      # tema di bawah ini tidak ikut ranking
SENTIMENT_REVIEW_THRESHOLD = 0.7  # IndoBERT di bawah ini dikirim ke AI
REVIEW_SAMPLE_RATE = 0.05         # sampel acak confidence tinggi yang tetap dicek AI
DEDUP_PER_AUTHOR = False          # True = retweet dari author berbeda tetap dihitung (ukur penyebar)

# Pending training data (dipakai bersama semua topik, tiap baris bawa field project)
PENDING_PATHS = {
    "ner": "/content/drive/MyDrive/pending_ner_data.jsonl",
    "sentimen": "/content/drive/MyDrive/pending_sentiment_data.jsonl",
    "tema": "/content/drive/MyDrive/pending_tema_data.jsonl",
}
SAVE_PATH_AFTER_THEME = f"/content/drive/MyDrive/df_clean_after_theme_{PROJECT}.pkl"


def save_pending(task, rows):
    """Append ke pending. Hanya label dari AI/manual; output mentah model ditolak."""
    path = PENDING_PATHS[task]
    rows = [r for r in rows if r.get("text") and r.get("label_source") in ("gemini", "manual")]
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({**r, "task": task, "project": PROJECT, "run_date": str(date.today())},
                               ensure_ascii=False) + "\n")
    print(f"[pending] {task}: {len(rows)} baris ditulis ke {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 5. Konfigurasi topik dari Colab AI

In [25]:
TOPIC_CONFIG_PATH = f"/content/drive/MyDrive/topic_config_{PROJECT}.json"
REGENERATE_TOPIC_CONFIG = False   # True = minta AI susun ulang (tema bisa berubah)

TOPIC_CONFIG_PROMPT = f"""Kamu analis media sosial Indonesia. Topik yang dipantau: "{TOPIC}".

Susun konfigurasi untuk mengklasifikasi cuitan tentang topik ini:
- themes: 2-4 tema yang tidak saling tumpang tindih. Nama snake_case, deskripsi berisi frasa
  yang biasa muncul di cuitan, termasuk singkatan dan julukan. Jangan buat tema "lainnya".
- keywords: kata atau frasa huruf kecil yang hampir selalu muncul di cuitan yang relevan,
  termasuk singkatan, julukan, dan salah ketik yang umum.
- aliases: nama lain tokoh, tempat, dan instansi (key huruf kecil) -> nama baku.
- media_accounts: username akun resmi atau media yang khusus terkait topik ini, huruf kecil tanpa @.

Jawab HANYA JSON dengan field themes (objek nama -> deskripsi), keywords (list),
aliases (objek), media_accounts (list)."""

if os.path.exists(TOPIC_CONFIG_PATH) and not REGENERATE_TOPIC_CONFIG:
    with open(TOPIC_CONFIG_PATH, encoding="utf-8") as f:
        topic_config = json.load(f)
    print(f"[INFO] Konfigurasi topik dibaca dari {TOPIC_CONFIG_PATH}")
else:
    topic_config = ask_json(TOPIC_CONFIG_PROMPT)
    if topic_config is None:
        raise RuntimeError("Colab AI gagal menyusun konfigurasi topik. Jalankan ulang cell ini.")
    with open(TOPIC_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(topic_config, f, ensure_ascii=False, indent=2)
    print(f"[INFO] Konfigurasi topik dari AI disimpan ke {TOPIC_CONFIG_PATH}")

THEME_DESCRIPTIONS = {**topic_config["themes"], "lainnya": "topik di luar kategori di atas"}
THEMES = list(THEME_DESCRIPTIONS)
RELEVANT_THEMES = [t for t in THEMES if t != "lainnya"]
KW_RELEVAN = [k.lower() for k in topic_config.get("keywords", [])]
ENTITY_ALIASES = {k.lower(): v for k, v in topic_config.get("aliases", {}).items()}
EXTRA_MEDIA_ACCOUNTS = {a.lower().lstrip("@") for a in topic_config.get("media_accounts", [])}

# Baris tanpa satu pun keyword langsung "lainnya" (hemat AI + GLiNER). Tanpa keyword = semua baris ke AI.
RE_RELEVAN = re.compile(
    r"(?<!\w)(?:" + "|".join(map(re.escape, sorted(set(KW_RELEVAN), key=len, reverse=True))) + r")(?!\w)"
) if KW_RELEVAN else None


def canonical(names):
    """Nama lain -> nama baku (ENTITY_ALIASES), sisanya apa adanya."""
    return [ENTITY_ALIASES.get(n.strip().lower(), n.strip()) for n in names]


print(json.dumps(topic_config, ensure_ascii=False, indent=2))

[INFO] Konfigurasi topik dibaca dari /content/drive/MyDrive/topic_config_prabowo_vladivostok.json
{
  "themes": {
    "diplomasi_bilateral": "hubungan diplomatik, kerja sama bilateral, geopolitik, hubungan indo-rusia, kepentingan nasional, aliansi, posisi indonesia",
    "agenda_pertemuan": "isi pembicaraan, hasil pertemuan, agenda utama, kesepakatan, tujuan kunjungan, diskusi strategis, poin penting",
    "reaksi_publik": "tanggapan publik, opini masyarakat, pro kontra, dukungan, kritik, reaksi netizen, pandangan ahli, sentimen media sosial",
    "narasi_media": "liputan media, berita terkini, sorotan pers, framing berita, pemberitaan media, analisis jurnalis, headliner"
  },
  "keywords": [
    "prabowo",
    "presiden prabowo",
    "prabs",
    "putin",
    "vladimir putin",
    "vladivostok",
    "rusia",
    "kunjungan",
    "bertemu",
    "pertemuan",
    "kerjasama",
    "diplomasi",
    "indo-rusia",
    "indonesia rusia",
    "geopolitik",
    "kremlin",
    "sammit",
    "ktt

# Step 1 — Load Data dari Google Sheets

Output: tabel mentah + daftar kolom, buat konfirmasi data ke-load dengan benar.


In [26]:
def xlsx_url(link):
    """Link Google Sheets/Drive -> url download .xlsx. Selain itu (path lokal) dipakai apa adanya."""
    match = re.search(r"/d/([\w-]+)", link)
    if not match:
        return link
    if "drive.google.com" in link:
        return f"https://drive.google.com/uc?export=download&id={match.group(1)}"
    return f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"


def load_raw_data(link):
    try:
        df = pd.read_excel(xlsx_url(link), header=1)
    except ValueError as e:
        raise ValueError(f"Gagal baca {link}. Pastikan akses file 'Anyone with the link'.") from e
    df.columns = df.columns.str.strip()
    return df[EXPECTED_COLS]   # KeyError menyebut kolom yang hilang


# Duplikat post lintas file dibuang di Step 3 (dedup_key).
df_raw = pd.concat([load_raw_data(s).assign(source_file=f"file_{i}") for i, s in enumerate(SOURCES, 1)],
                   ignore_index=True)
print(f"Data berhasil dimuat: {df_raw.shape}")
display(df_raw.head())

Data berhasil dimuat: (66891, 13)


,No,Type,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers,Retweeted,Favourited,source_file
0,1,mention,"Netflix Rilis Skenario Sang Jenderal, Film Dok...",rmol news logo Film dokumenter Skenario Sang J...,2026-09-22 15:45:00,https://rmol.id/hiburan/read/2026/09/22/722557...,News,Negative,rmol.id,0,0,0,file_1
1,2,rt,NaN,"RT Negara hancur, Aturan baru OJK: Danantara, ...",2026-09-22 15:32:35,https://twitter.com/web/statuses/2102315108575...,Twitter,Negative,@iwantjiminp,5030,0,0,file_1
2,3,mention,Singapura Luncurkan Gerakan Baca 15 Menit Seha...,Presiden Prabowo Subianto berjabat tangan dan ...,2026-09-22 15:31:00,https://www.kompas.com/tren/read/2026/09/22/15...,News,Positive,www.kompas.com,0,0,0,file_1
3,4,mention,Singapura Luncurkan Gerakan Baca 15 Menit Seha...,Presiden Prabowo Subianto berjabat tangan dan ...,2026-09-22 15:31:00,https://www.kompas.com/tren/read/2026/09/22/15...,News,Positive,www.kompas.com,0,0,0,file_1
4,5,mention,Menkeu Ungkap Surplus BI Jadi Sumber Dana Luna...,Indonesia&#39;s Finance Minister Suahasil Naza...,2026-09-22 15:29:22,https://www.cnnindonesia.com/ekonomi/202609221...,News,Negative,www.cnnindonesia.com,0,0,0,file_1


# Step 2 - Filter akun media (satu-satunya definisi + eksekusi)


In [27]:
for col in (COL_FOLLOWERS, COL_RETWEETED, COL_FAVOURITED):
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce").fillna(0)

is_media = pd.Series([is_media_account(a, p, extra_accounts=EXTRA_MEDIA_ACCOUNTS)
                      for a, p in zip(df_raw[COL_AUTHOR_ID], df_raw[COL_MEDIA])], index=df_raw.index)

print(f"[INFO] {is_media.sum()} baris media dibuang, {(~is_media).sum()} baris non-media lanjut")
display(df_raw.assign(is_media=is_media)[[COL_AUTHOR_ID, COL_MEDIA, "is_media"]]
        .drop_duplicates(subset=COL_AUTHOR_ID).head(20))

df_raw = df_raw[~is_media].reset_index(drop=True)

[INFO] 28691 baris media dibuang, 38200 baris non-media lanjut


,Author,Media,is_media
0,rmol.id,News,True
1,@iwantjiminp,Twitter,False
2,www.kompas.com,News,True
4,www.cnnindonesia.com,News,True
5,@wanzhouvers,Twitter,False
6,economy.okezone.com,News,True
8,index.okezone.com,News,True
10,@moodmasyarakat,Twitter,False
11,wartaekonomi.co.id,News,True
16,nasional.tvrinews.com,News,True


# Step 3 - cleaning text

In [31]:
RE_RT_PREFIX = re.compile(r"^\s*RT\s*@[A-Za-z0-9_]+\s*:\s*", flags=re.IGNORECASE)
RE_ELONGATION = re.compile(r"(.)\1{2,}")
EMOJI_LANG = "id" if "id" in emoji.LANGUAGES else "en"


def strip_noise(text):
    """Buang prefix RT, emoji -> kata, 'mantaaap' -> 'mantaap'. Dipakai untuk BERT dan NER."""
    text = RE_RT_PREFIX.sub("", text)
    text = emoji.demojize(text, language=EMOJI_LANG).replace("_", " ").replace(":", " ")
    return RE_ELONGATION.sub(r"\1\1", text)


def build_raw_text(headline, mentions):
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if headline.lower() in ("", "nan", mentions.lower()):
        return mentions or headline
    return f"{headline}. {mentions}" if mentions else headline


df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(df_raw[COL_HEADLINE], df_raw[COL_MENTIONS])]
noise_free = df_raw["text_raw_combined"].map(strip_noise)
df_raw["text_clean"] = noise_free.progress_apply(clean_for_bert)   # clean_for_bert sudah merapikan spasi
df_raw["text_ner"] = noise_free.apply(clean_for_ner)                # GLiNER: case & slang dipertahankan

df_clean = df_raw[df_raw["text_clean"].str.split().str.len() >= 3].copy()
print(f"[INFO] Buang {len(df_raw) - len(df_clean)} baris kosong/terlalu pendek")

df_clean["dedup_key"] = (df_clean["text_clean"].str.lower()
                         .str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" "))
before = len(df_clean)
df_clean = (df_clean.drop_duplicates(subset=["dedup_key", COL_AUTHOR_ID] if DEDUP_PER_AUTHOR else ["dedup_key"])
            .drop(columns="dedup_key").reset_index(drop=True))
print(f"[INFO] Buang {before - len(df_clean)} duplikat (per author: {DEDUP_PER_AUTHOR})")
print(f"[OK] {len(df_clean)} baris tersisa setelah cleaning")

display(df_clean[["text_raw_combined", "text_clean", "text_ner"]].head(5))

  0%|          | 0/38200 [00:00<?, ?it/s]

[INFO] Buang 1670 baris kosong/terlalu pendek
[INFO] Buang 13899 duplikat (per author: False)
[OK] 22631 baris tersisa setelah cleaning


,text_raw_combined,text_clean,text_ner
0,"RT Negara hancur, Aturan baru OJK: Danantara, ...","RT Negara hancur, Aturan baru OJK Danantara, B...","RT Negara hancur, Aturan baru OJK Danantara, B..."
1,"RT ""Prabowo NPD this NPD that.."" . . . .. tapi...","RT ""Prabowo NPD this NPD that.."" . . . .. tapi...","RT ""Prabowo NPD this NPD that.."" . . . .. tapi..."
2,Pakar Hukum UNPAD Romli Atmasasmita: Presiden ...,Pakar Hukum UNPAD Romli Atmasasmita Presiden P...,Pakar Hukum UNPAD Romli Atmasasmita Presiden P...
3,RT P2G Desak MBG Dihentikan: Guru Jangan Terus...,RT P2G Desak MBG Dihentikan Guru Jangan Terus ...,RT P2G Desak MBG Dihentikan Guru Jangan Terus ...
4,"emang prabowo sialan, mbg sialan [RE kenhans03]","memang prabowo sialan, mbg sialan [RE kenhans03]","emang prabowo sialan, mbg sialan [RE kenhans03]"


# Step 4A — Load GLiNER

In [38]:
import torch
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME).to("cuda" if torch.cuda.is_available() else "cpu")

# Potongan dihitung dalam token GLiNER (tanda baca = 1 token), bukan kata, supaya tidak ada yang terpotong.
GLINER_CHUNK_TOKENS = gliner_model.config.max_len
GLINER_CHUNK_OVERLAP = 20   # token yang diulang di awal potongan berikutnya, supaya entitas di batas tidak terbelah

print(f"GLiNER siap di {'GPU' if torch.cuda.is_available() else 'CPU'} | "
      f"potongan {GLINER_CHUNK_TOKENS} token")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

GLiNER siap di GPU | potongan 384 token


# Step 4B — GLiNER extract entitas mentah

In [39]:
RE_GLINER_TOKEN = re.compile(r"\w+(?:[-_]\w+)*|\S")   # sama dengan WhitespaceTokenSplitter bawaan GLiNER


def gliner_chunks(text):
    """Potong teks per GLINER_CHUNK_TOKENS token (tumpang GLINER_CHUNK_OVERLAP), teks asli tetap utuh."""
    toks = list(RE_GLINER_TOKEN.finditer(text))
    step = GLINER_CHUNK_TOKENS - GLINER_CHUNK_OVERLAP
    return [text[toks[i].start():toks[min(i + GLINER_CHUNK_TOKENS, len(toks)) - 1].end()]
            for i in range(0, max(len(toks) - GLINER_CHUNK_OVERLAP, 1), step)]


def gliner_extract_entities(text):
    """Return (entities per label dalam bentuk persis seperti di teks, confidence terendah)."""
    result = {lab: [] for lab in GLINER_LABELS}
    scores = []
    for chunk in gliner_chunks(text):
        for e in gliner_model.predict_entities(chunk, GLINER_LABELS, threshold=GLINER_THRESHOLD):
            if e["label"] in result and e["text"] not in result[e["label"]]:
                result[e["label"]].append(e["text"])
                scores.append(e["score"])
    return result, min(scores, default=None)

# Step 4C — Gemini: tema + validasi NER (build_validation_prompt, entities_dict_to_spans)

In [40]:
THEME_LIST_STR = "\n".join(f"- {t}: {THEME_DESCRIPTIONS[t]}" for t in THEME_DESCRIPTIONS)


def build_validation_prompt(batch_items):
    """batch_items: list of (text, gliner_entities_dict)"""
    joined = "\n".join(
        f'{i}. Teks: "{text}"\n'
        f'   Prediksi GLiNER -> lokasi: {ents["lokasi"]}, instansi: {ents["instansi"]}, tokoh: {ents["tokoh"]}'
        for i, (text, ents) in enumerate(batch_items, 1)
    )
    return f"""Kamu classifier tema dan validator NER untuk cuitan bahasa Indonesia tentang: {TOPIC}.

Daftar tema:
{THEME_LIST_STR}

Untuk setiap item, tentukan tema dan periksa prediksi GLiNER: pindahkan entitas yang salah
kategori, tambahkan yang terlewat, hapus yang bukan entitas asli.
- Tulis entitas persis seperti di teks (ejaan, singkatan, huruf besar-kecil). Jangan dinormalisasi atau dilengkapi.
- Jabatan tanpa nama bukan tokoh.

Item:
{joined}

Jawab HANYA JSON array berisi tepat {len(batch_items)} objek, masing-masing dengan field:
index (nomor item), primary_theme (salah satu nama tema di atas), confidence (0-1),
lokasi, instansi, tokoh (list string)."""


def entities_dict_to_spans(text, entities):
    """{"lokasi": [surface,...], ...} -> ([[start, end, label], ...], [nama tidak ketemu]).
    Semua kemunculan, case-insensitive, kata utuh. Nama panjang duluan."""
    pairs = sorted({(n.strip(), lab) for lab, names in entities.items() for n in names if n and n.strip()},
                   key=lambda p: len(p[0]), reverse=True)
    spans, missing = [], []
    for name, label in pairs:
        matches = list(re.finditer(r"(?<!\w)" + re.escape(name) + r"(?!\w)", text, re.IGNORECASE))
        if not matches:
            missing.append(name)
        for m in matches:
            if not any(m.start() < e and s < m.end() for s, e, _ in spans):
                spans.append([m.start(), m.end(), label])
    return sorted(spans), missing

# Step 4D — Eksekusi

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# --- Kecepatan vs kelengkapan ---
# Estimasi waktu ~= (baris / BATCH_SIZE_CLASSIFY) x (detik per batch + SLEEP_BETWEEN_CALLS) / AI_WORKERS
# "detik per batch" lihat di bar tqdm (s/it) setelah beberapa batch jalan.
AI_WORKERS = 2        # request AI paralel. 1 = paling aman. Makin besar makin cepat, tapi kuota lebih cepat habis
                     # dan bisa kena rate limit (429) -> otomatis berhenti + checkpoint, turunkan angkanya lalu lanjut.
MAX_AI_ROWS = None   # None = semua baris. Angka = ambil sampel acak sebanyak ini (GLiNER + AI ikut lebih cepat).
                     # Trade-off: sisanya tidak punya tema -> ranking Step 5 dari sampel, author dengan sedikit
                     # post bisa terlewat. Jumlah post/engagement ikut mengecil, urutan ranking kurang lebih sama.
CHECKPOINT_PATH = f"/content/drive/MyDrive/checkpoint_4d_{PROJECT}.pkl"
CHECKPOINT_EVERY = 10   # batch


def save_checkpoint(next_b):
    pd.to_pickle({"df": df_clean, "gliner": gliner_results, "review_idx": review_idx, "next_b": next_b,
                  "ner": ner_training_data, "tema": tema_training_data, "n_skip": n_skip_missing},
                 CHECKPOINT_PATH)


def validate_batch(b):
    """Satu request AI (jalan di thread). Return (index df, hasil per item atau None)."""
    idx_batch = review_idx[b * BATCH_SIZE_CLASSIFY:(b + 1) * BATCH_SIZE_CLASSIFY]
    items = [(df_clean.at[i, "text_ner"], gliner_results[i]["entities"]) for i in idx_batch]
    # hasil dicocokkan lewat field 'index', BUKAN posisi
    return idx_batch, ask_json(build_validation_prompt(items),
                               parse=lambda p: order_by_index(p, len(items)), retry=6)


if os.path.exists(CHECKPOINT_PATH):
    ckpt = pd.read_pickle(CHECKPOINT_PATH)
    df_clean, gliner_results, review_idx, start_b = ckpt["df"], ckpt["gliner"], ckpt["review_idx"], ckpt["next_b"]
    ner_training_data, tema_training_data, n_skip_missing = ckpt["ner"], ckpt["tema"], ckpt["n_skip"]
    print(f"[RESUME] Lanjut dari batch {start_b} (checkpoint {CHECKPOINT_PATH})")
else:
    df_clean[["theme", "theme_confidence", "ner_source"]] = None
    df_clean[["ner_failed", "from_prefilter", "gliner_corrected"]] = False
    for lab in GLINER_LABELS:
        df_clean[f"entities_{lab}"] = [[] for _ in range(len(df_clean))]

    # tahap 1: keyword prefilter -> tanpa satu pun keyword relevan langsung "lainnya"
    if RE_RELEVAN:
        irrelevant = ~df_clean["text_clean"].str.lower().str.contains(RE_RELEVAN)
        df_clean.loc[irrelevant, ["theme", "theme_confidence", "from_prefilter"]] = ["lainnya", 1.0, True]
    review_idx = df_clean.index[df_clean["theme"].isna()].tolist()
    print(f"[INFO] Prefilter: {df_clean['from_prefilter'].sum()} baris 'lainnya', {len(review_idx)} lanjut ke GLiNER + AI")
    if MAX_AI_ROWS and len(review_idx) > MAX_AI_ROWS:
        review_idx = sorted(random.sample(review_idx, MAX_AI_ROWS))
        print(f"[WARN] Sampling: hanya {MAX_AI_ROWS} baris diproses, sisanya tanpa tema (ranking dari sampel)")

    # tahap 2: GLiNER di baris yang lolos prefilter
    gliner_results = {}
    for idx in tqdm(review_idx, desc="GLiNER NER"):
        ents, min_conf = gliner_extract_entities(df_clean.at[idx, "text_ner"])
        gliner_results[idx] = {"entities": ents, "min_conf": min_conf}
        for lab in GLINER_LABELS:
            df_clean.at[idx, f"entities_{lab}"] = canonical(ents[lab])
        df_clean.at[idx, "ner_source"] = "gliner"

    ner_training_data, tema_training_data = [], []
    n_skip_missing = 0
    start_b = 0
    save_checkpoint(0)   # GLiNER juga lama, jangan diulang kalau putus

# tahap 3: AI tentukan tema + validasi NER, AI_WORKERS batch sekaligus
n_batches = -(-len(review_idx) // BATCH_SIZE_CLASSIFY)
print(f"[INFO] Sisa {n_batches - start_b} batch, {AI_WORKERS} paralel")
pbar = tqdm(total=n_batches, initial=start_b, desc="Validate NER+Tema")
finished = True

with ThreadPoolExecutor(AI_WORKERS) as pool:
    for b0 in range(start_b, n_batches, AI_WORKERS):
        next_b = min(b0 + AI_WORKERS, n_batches)
        try:
            done = list(pool.map(validate_batch, range(b0, next_b)))
        except Exception as e:
            if is_quota_error(e):
                save_checkpoint(b0)   # batch di putaran ini diulang saat lanjut
                print(f"[STOP] Kuota habis / rate limit di batch {b0}/{n_batches}. "
                      "Tunggu, (turunkan AI_WORKERS), lalu jalankan ulang cell ini.")
                finished = False
                break
            raise

        for idx_batch, results in done:
            if results is None:
                # tema tetap None (terhitung belum diproses), entity dari GLiNER, tidak masuk training
                print("[ERROR] Batch validasi gagal total. Entity tetap dari GLiNER, baris TIDAK masuk training.")
                df_clean.loc[idx_batch, "ner_failed"] = True
                continue

            for r, df_idx in zip(results, idx_batch):
                theme = r.get("primary_theme")
                theme = theme if theme in THEMES else "lainnya"
                conf = float(r.get("confidence", 0.5))
                df_clean.at[df_idx, "theme"] = theme
                df_clean.at[df_idx, "theme_confidence"] = conf
                tema_training_data.append({"text": df_clean.at[df_idx, "text_clean"], "label": theme,
                                           "confidence": conf, "label_source": "gemini"})

                ents = {lab: [x for x in (r.get(lab) or []) if isinstance(x, str)] for lab in GLINER_LABELS}
                for lab in GLINER_LABELS:
                    df_clean.at[df_idx, f"entities_{lab}"] = canonical(ents[lab])
                df_clean.at[df_idx, "ner_source"] = "gemini"

                gl = gliner_results[df_idx]["entities"]
                df_clean.at[df_idx, "gliner_corrected"] = any(
                    sorted(x.lower() for x in gl[lab]) != sorted(x.lower() for x in ents[lab]) for lab in GLINER_LABELS)

                text_ner = df_clean.at[df_idx, "text_ner"]
                spans, missing = entities_dict_to_spans(text_ner, ents)
                if missing:
                    n_skip_missing += 1   # label parsial = racun training, jangan dipakai
                    continue
                ner_training_data.append({
                    "text": text_ner, "entities": spans, "label_source": "gemini",
                    "review_reason": "tema_ambigu", "model_version": GLINER_MODEL_NAME,
                    "last_confidence": gliner_results[df_idx]["min_conf"],
                })

        pbar.update(next_b - b0)
        if next_b // CHECKPOINT_EVERY != b0 // CHECKPOINT_EVERY:
            save_checkpoint(next_b)
        time.sleep(SLEEP_BETWEEN_CALLS)
pbar.close()

if finished:
    # training data ditulis sekali di sini (bukan tiap resume, supaya tidak dobel)
    save_pending("ner", ner_training_data)
    save_pending("tema", tema_training_data)
    os.remove(CHECKPOINT_PATH)

validated = df_clean["ner_source"] == "gemini"
if validated.any():
    rate = df_clean.loc[validated, "gliner_corrected"].mean()
    print(f"\n[INFO] correction_rate GLiNER: {100 * rate:.1f}% dari {validated.sum()} baris tervalidasi")
print(f"[INFO] {n_skip_missing} baris tidak masuk training (entity AI tidak ditemukan di teks)")

n_unprocessed = df_clean["theme"].isna().sum()
if n_unprocessed:
    print(f"[WARN] {n_unprocessed} baris BELUM punya tema (sampling / kuota habis / batch gagal). "
          "Laporan di bawah TIDAK lengkap.")

print("\nDistribusi tema:")
display(df_clean["theme"].value_counts(dropna=False))

print("\nContoh hasil klasifikasi + NER:")
display(df_clean.loc[df_clean["theme"].isin(RELEVANT_THEMES),
        ["text_clean", "theme", "ner_source", "entities_lokasi", "entities_instansi", "entities_tokoh"]].head(10))

df_clean.to_pickle(SAVE_PATH_AFTER_THEME)
print(f"[OK] df_clean tersimpan ke {SAVE_PATH_AFTER_THEME}. Total API call: {api_call_count}")

[RESUME] Lanjut dari batch 36 (checkpoint /content/drive/MyDrive/checkpoint_4d_prabowo_vladivostok.pkl)
[INFO] Sisa 260 batch, 2 paralel


Validate NER+Tema:  12%|#2        | 36/296 [00:00<?, ?it/s]

# Step 5 - Ranking Top Author per Tema

Hanya akun non-media (sudah difilter Step 2). Skor = post count + engagement + followers.

In [43]:
processed = df_clean["theme"].notna()
relevant = processed & (df_clean["theme"] != "lainnya")
confident = df_clean["theme_confidence"].astype(float) >= RANKING_MIN_CONFIDENCE

df_topic = df_clean[relevant & confident].copy()
print(f"[INFO] {len(df_topic)} baris ikut ranking, "
      f"{(relevant & ~confident).sum()} baris dibuang karena tema ragu (< {RANKING_MIN_CONFIDENCE})")
if (~processed).sum():
    print(f"[WARN] {(~processed).sum()} baris belum diproses. Ranking PARSIAL.")

df_topic["engagement"] = df_topic[COL_FAVOURITED] + df_topic[COL_RETWEETED]

agg = (df_topic.groupby(["theme", COL_AUTHOR_ID])
       .agg(jumlah_post=("engagement", "size"),
            total_engagement=("engagement", "sum"),
            followers=(COL_FOLLOWERS, "max"))
       .reset_index()
       .rename(columns={COL_AUTHOR_ID: "author_id"}))
# tiap metrik dinormalisasi ke max-nya dalam tema yang sama
metrics = agg[["jumlah_post", "total_engagement", "followers"]].astype(float)
norm = metrics / metrics.groupby(agg["theme"]).transform("max").replace(0, 1)
agg["score"] = norm @ [WEIGHT_POST_COUNT, WEIGHT_ENGAGEMENT, WEIGHT_FOLLOWERS]

df_top_authors = (agg.sort_values(["theme", "score"], ascending=[True, False])
                  .groupby("theme").head(TOP_N_AUTHORS_PER_THEME).reset_index(drop=True))
df_top_authors["rank"] = df_top_authors.groupby("theme").cumcount() + 1

for theme in RELEVANT_THEMES:
    sub = df_top_authors[df_top_authors["theme"] == theme]
    if sub.empty:
        print(f"[INFO] Tidak ada data untuk tema '{theme}'")
        continue
    print(f"\n=== Top Author: {theme} ===")
    display(sub[["rank", "author_id", "jumlah_post", "total_engagement", "followers", "score"]])

[INFO] 843 baris ikut ranking, 0 baris dibuang karena tema ragu (< 0.6)
[WARN] 10386 baris belum diproses. Ranking PARSIAL.

=== Top Author: diplomasi_bilateral ===


,rank,author_id,jumlah_post,total_engagement,followers,score
10,1,@puputirawa5557,15,0,414,0.600048
11,2,@LambeSahamjja,1,41,85254,0.299951
12,3,@straits_times,1,2,1285133,0.202195
13,4,@thejakartaglobe,1,0,457347,0.093381
14,5,Dedi,2,0,16,0.080002
15,6,bahlil.lahadalia,1,0,142400,0.056621
16,7,@WeKa_Ronin,1,2,8247,0.053158
17,8,@Aerionreporter,1,2,101,0.052207
18,9,@AndrewHolmes01,1,1,1159,0.046233
19,10,euzli,1,0,10200,0.041191



=== Top Author: agenda_pertemuan ===


,rank,author_id,jumlah_post,total_engagement,followers,score
0,1,@Heraloebss,1,19,175472,0.700000
1,2,@daniel_dvd0,2,0,19,0.600016
2,3,@LambeSahamjja,1,0,85326,0.372940
3,4,@99propaganda,1,1,5144,0.317555
4,5,@ter_sendat,1,1,29,0.313183
5,6,@kabarrnusantara,1,1,0,0.313158
6,7,@vooxid_,1,1,0,0.313158
7,8,@renjunhuanm,1,0,1793,0.301533
8,9,@nurasmi69,1,0,1587,0.301357
9,10,@AditiyaP25269,1,0,1331,0.301138



=== Top Author: reaksi_publik ===


,rank,author_id,jumlah_post,total_engagement,followers,score
30,1,@FerryFe71749796,6,6,70,0.625867
31,2,@JokowiP3n1pu,4,13,972,0.456098
32,3,@hamas191,4,2,2053,0.408754
33,4,@AzzamIzzulhaq,1,58,390924,0.375450
34,5,@geloraco,1,53,687052,0.373178
35,6,@BudiBukanIntel,1,58,114782,0.357473
36,7,@ruhutsitompul,1,14,2304029,0.310345
37,8,@WeKa_Ronin,3,2,8247,0.309158
38,9,@Megatop99,3,0,6279,0.300409
39,10,Jhony,3,0,37,0.300002



=== Top Author: narasi_media ===


,rank,author_id,jumlah_post,total_engagement,followers,score
20,1,@puputirawa5557,24,0,414,0.600020
21,2,@bolpoinmer4h,1,202,3,0.275000
22,3,@INDONESIAinLOVE,1,7,3164734,0.183663
23,4,@zivamela,4,2,27,0.102477
24,5,@SinarOnline,1,0,1103607,0.077308
25,6,@Kimberley_PS08,3,0,4176,0.075198
26,7,@Ochi_Queen09_1,2,16,25583,0.071015
27,8,Arki Rifazka,2,0,16554,0.050785
28,9,@ronald05001773,2,0,440,0.050021
29,10,ResPublicaProject,2,0,331,0.050016


# Step 6A - Load IndoBERT sentimen

In [44]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

indobert_tokenizer = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
indobert_model = AutoModelForSequenceClassification.from_pretrained(INDOBERT_MODEL_NAME).eval()

# label map dari config model, bukan hardcode (urutan label tiap model beda)
LABEL_NORMALIZE = {"positive": "positif", "negative": "negatif", "neutral": "netral"}
INDOBERT_LABEL_MAP = {int(i): LABEL_NORMALIZE.get(str(l).lower(), str(l).lower())
                      for i, l in indobert_model.config.id2label.items()}
print("[INFO] Label map dari config model:", INDOBERT_LABEL_MAP)
if not {"positif", "negatif", "netral"} <= set(INDOBERT_LABEL_MAP.values()):
    raise ValueError("id2label tidak dikenali (mis. LABEL_0). Cek model card, isi INDOBERT_LABEL_MAP manual.")


def indobert_predict_sentiment(text):
    inputs = indobert_tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        probs = torch.softmax(indobert_model(**inputs).logits, dim=1)[0]
    pred_id = int(torch.argmax(probs))
    return INDOBERT_LABEL_MAP[pred_id], float(probs[pred_id])

print("IndoBERT siap.")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[INFO] Label map dari config model: {0: 'positif', 1: 'netral', 2: 'negatif'}
IndoBERT siap.


# Step 6B - IndoBERT per post, lalu digabung per author

In [45]:
def aggregate_author_sentiment(labels):
    """labels: label sentimen per post (tidak kosong).
    kontroversi = campuran positif & negatif signifikan. netral = mayoritas netral / tanpa sikap."""
    counts = Counter(labels)
    pos, neg, net = (counts[k] / len(labels) for k in ("positif", "negatif", "netral"))
    if pos >= 0.3 and neg >= 0.3:
        return "kontroversi", max(pos, neg)
    if net >= 0.5 or pos == neg:
        return "netral", net
    return ("positif", pos) if pos > neg else ("negatif", neg)


def label_sentiment_batch(items):
    """items: list of (text, theme). Return label per item (None kalau gagal)."""
    numbered = "\n".join(f'{i}. Tema: {th.replace("_", " ")}\n   Teks: "{t}"'
                         for i, (t, th) in enumerate(items, 1))
    prompt = f"""Tentukan sikap penulis setiap cuitan terhadap tema yang disebut:
positif (mendukung, memuji, optimis), negatif (mengkritik, menolak, pesimis),
atau netral (informasi tanpa sikap, atau sikap tidak jelas).

Item:
{numbered}

Jawab HANYA JSON array berisi tepat {len(items)} objek dengan field index dan label."""
    n = len(items)
    labels = ask_json(prompt, parse=lambda p: [x.get("label") for x in order_by_index(p, n)]) or [None] * n
    return [lab if lab in ("positif", "negatif", "netral") else None for lab in labels]


author_sentiment_results = {}
review_queue = {}   # (text, theme) -> (confidence, reason), dedup otomatis

for author_id, theme in zip(df_top_authors["author_id"], df_top_authors["theme"]):
    posts = (df_topic[(df_topic[COL_AUTHOR_ID] == author_id) & (df_topic["theme"] == theme)]
             .sort_values("engagement", ascending=False)   # post paling berpengaruh duluan
             ["text_clean"].dropna().tolist()[:MAX_POSTS_PER_AUTHOR_SUMMARY])
    if not posts:
        continue

    post_sentiments = [indobert_predict_sentiment(t) for t in posts]
    for text, (_, conf) in zip(posts, post_sentiments):
        if conf < SENTIMENT_REVIEW_THRESHOLD:
            review_queue.setdefault((text, theme), (conf, "low_conf"))
        elif random.random() < REVIEW_SAMPLE_RATE:
            review_queue.setdefault((text, theme), (conf, "sample"))

    label, confidence = aggregate_author_sentiment([lab for lab, _ in post_sentiments])
    author_sentiment_results[(author_id, theme)] = {
        "sentiment": label, "confidence": confidence, "post_sentiments": post_sentiments, "posts": posts,
    }

print(f"[INFO] IndoBERT selesai prediksi {len(author_sentiment_results)} author")

# label koreksi per post dari AI -> training data sentimen
sentiment_training_data = []
queue_items = list(review_queue.items())
print(f"[INFO] {len(queue_items)} post dikirim ke AI untuk label sentimen")

for b in range(0, len(queue_items), BATCH_SIZE_CLASSIFY):
    chunk = queue_items[b:b + BATCH_SIZE_CLASSIFY]
    try:
        labels = label_sentiment_batch([key for key, _ in chunk])
    except Exception as e:
        if is_quota_error(e):
            print(f"[STOP] Kuota habis saat label sentimen. {len(sentiment_training_data)} label tersimpan.")
            break
        raise
    sentiment_training_data += [
        {"text": text, "theme": theme, "label": label, "label_source": "gemini",
         "review_reason": reason, "model_version": INDOBERT_MODEL_NAME, "last_confidence": conf}
        for ((text, theme), (conf, reason)), label in zip(chunk, labels) if label
    ]
    time.sleep(SLEEP_BETWEEN_CALLS)

save_pending("sentimen", sentiment_training_data)

[INFO] IndoBERT selesai prediksi 40 author
[INFO] 29 post dikirim ke AI untuk label sentimen
[pending] sentimen: 29 baris ditulis ke /content/drive/MyDrive/pending_sentiment_data.jsonl


# Step 6C - Ringkasan naratif

In [46]:
NARRATIVE_FALLBACK = {"summary": "(gagal diringkas otomatis)", "reason": "error API", "disagree_reason": ""}


def generate_narrative(author_label, theme, texts, sentiment_label, sentiment_confidence):
    joined = "\n".join(f"- {t}" for t in texts)
    prompt = f"""Cuitan dari akun "{author_label}" tentang tema "{theme}":

{joined}

Model sentimen menilai sikap akun ini "{sentiment_label}" (confidence {sentiment_confidence:.2f}).

Jawab HANYA JSON dengan field:
- summary: 2-3 kalimat pandangan atau narasi utama akun ini soal tema tersebut
- reason: 1 kalimat kenapa sentimen "{sentiment_label}" masuk akal berdasarkan isi teks
- disagree_reason: alasan kalau menurutmu label sentimen itu salah, string kosong kalau setuju"""
    return ask_json(prompt, retry=3) or NARRATIVE_FALLBACK


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

results = []
quota_hit = False

for _, row in df_top_authors.iterrows():
    data = author_sentiment_results.get((row["author_id"], row["theme"]))
    if data is None:
        continue
    print(f"Generate narasi @{row['author_id']} | tema={row['theme']} | sentimen={data['sentiment']} ...")

    try:
        narrative = generate_narrative(row["author_id"], row["theme"], data["posts"],
                                       data["sentiment"], data["confidence"])
    except Exception as e:
        if is_quota_error(e):
            quota_hit = True
            break
        raise

    if narrative.get("disagree_reason"):
        print(f"  [FLAG] AI tidak setuju sentimen IndoBERT: {narrative['disagree_reason']}")

    results.append({
        "theme": row["theme"],
        "rank": int(row["rank"]),
        "author_id": row["author_id"],
        "score": round(row["score"], 2),
        "sentiment": data["sentiment"],
        "sentiment_confidence": round(data["confidence"], 2),
        "summary": narrative.get("summary", ""),
        "reason": narrative.get("reason", ""),
        "ai_disagree": narrative.get("disagree_reason", ""),
    })

df_summary = pd.DataFrame(results)

if quota_hit:
    print(f"\n[STOP] Berhenti di tengah karena kuota habis. {len(results)} author sudah beres.")
else:
    print(f"\n[DONE] Semua author beres. Total API call sesi ini: {api_call_count}")

print("\n=== Hasil akhir: ringkasan & sentimen per top author ===")
display(df_summary)

Generate narasi @@Heraloebss | tema=agenda_pertemuan | sentimen=netral ...
Generate narasi @@daniel_dvd0 | tema=agenda_pertemuan | sentimen=netral ...
Generate narasi @@LambeSahamjja | tema=agenda_pertemuan | sentimen=netral ...
Generate narasi @@99propaganda | tema=agenda_pertemuan | sentimen=positif ...
Generate narasi @@ter_sendat | tema=agenda_pertemuan | sentimen=netral ...
  [FLAG] AI tidak setuju sentimen IndoBERT: Sentimen seharusnya positif karena tweet ini menggunakan bahasa yang sangat mendukung dan antusias, seperti pujian terhadap kemampuan produksi mandiri dan penggunaan tagar seperti 'kamibersamapresiden' serta 'EnergiMandiri' yang jelas menunjukkan dukungan kuat terhadap kebijakan dan figur Presiden.
Generate narasi @@kabarrnusantara | tema=agenda_pertemuan | sentimen=netral ...
Generate narasi @@vooxid_ | tema=agenda_pertemuan | sentimen=netral ...
Generate narasi @@renjunhuanm | tema=agenda_pertemuan | sentimen=netral ...
Generate narasi @@nurasmi69 | tema=agenda_pert

,theme,rank,author_id,score,sentiment,sentiment_confidence,summary,reason,ai_disagree
0,agenda_pertemuan,1,@Heraloebss,0.70,netral,1.00,"Akun ini menyampaikan pandangan Seskab Teddy Indra Wijaya mengenai agenda pendidikan Presiden Prabowo, yang meliputi pendidikan gratis dari SD hingga perguruan tinggi, perbaikan fasilitas sekolah, serta peningkatan kesejahteraan dan kompetensi guru. Inisiatif ini dipandang sebagai investasi penting bagi masa depan bangsa Indonesia.","Sentimen 'netral' masuk akal karena akun ini hanya mengutip pernyataan seorang pejabat tanpa menambahkan opini, penilaian, atau komentar pribadi yang menunjukkan keberpihakan atau ketidaksetujuan.",
1,agenda_pertemuan,2,@daniel_dvd0,0.60,netral,1.00,Akun ini menyampaikan kutipan dari Presiden Prabowo yang menekankan pentingnya kemandirian pangan dan energi bagi Indonesia agar tidak bergantung pada negara lain. Cuitan tersebut secara langsung mengutip pernyataan yang disampaikan oleh Presiden di Gontor.,"Sentimen netral masuk akal karena akun tersebut hanya melaporkan atau mengutip secara langsung pernyataan Presiden Prabowo tanpa menambahkan opini, penilaian, atau emosi pribadi.",
2,agenda_pertemuan,3,@LambeSahamjja,0.37,netral,1.00,"Akun ini melaporkan hasil pertemuan mengenai perkembangan ekonomi, utamanya pelunasan utang obligasi BLBI sebesar Rp 218,3 triliun kepada Bank Indonesia menggunakan dana surplus BI. Meskipun demikian, pemerintah akan terus menagih piutang BLBI dari debitur obligor, sebuah langkah yang disambut baik oleh Prabowo yang juga berpesan tentang pentingnya kemandirian pangan dan energi.",Sentimen 'netral' masuk akal karena cuitan ini menyajikan fakta-fakta mengenai pelunasan utang obligasi BLBI dan kelanjutan penagihan piutang tanpa menunjukkan opini atau keberpihakan akun tersebut.,
3,agenda_pertemuan,4,@99propaganda,0.32,positif,1.00,"Akun ini sangat mendukung rencana Presiden Prabowo Subianto untuk menerapkan teknologi Coal-to-Liquids (CTL) di Indonesia. Cuitan tersebut membantah keraguan terhadap gagasan Prabowo, menunjukkan keberhasilan implementasi CTL di China, dan mengaitkannya dengan penguatan ketahanan serta kemandirian energi nasional Indonesia melalui pemanfaatan cadangan batu bara yang melimpah.","Sentimen 'positif' masuk akal karena akun ini secara eksplisit memuji dan membela ide Presiden Prabowo, menggarisbawahi potensi positifnya untuk ketahanan energi nasional dan menepis anggapan bahwa ide tersebut tidak realistis.",
4,agenda_pertemuan,5,@ter_sendat,0.31,netral,1.00,Akun ini menyoroti kekhawatiran Presiden Prabowo terhadap risiko ketergantungan impor energi. Mereka memuji keberhasilan Indonesia dalam memproduksi solar dari kelapa sawit sebagai bentuk kemandirian dan mengindikasikan kelanjutan langkah serupa dengan batu bara. Cuitan ini secara kuat mendukung visi Presiden untuk menjaga ketahanan dan kemandirian energi nasional.,"Model mungkin menganggap tweet ini netral karena utamanya melaporkan pernyataan Presiden dan mendeskripsikan fakta tentang keberhasilan masa lalu serta rencana masa depan terkait strategi energi nasional, yang dapat diinterpretasikan sebagai penyampaian informasi objektif.","Sentimen seharusnya positif karena tweet ini menggunakan bahasa yang sangat mendukung dan antusias, seperti pujian terhadap kemampuan produksi mandiri dan penggunaan tagar seperti 'kamibersamapresiden' serta 'EnergiMandiri' yang jelas menunjukkan dukungan kuat terhadap kebijakan dan figur Presiden."
5,agenda_pertemuan,6,@kabarrnusantara,0.31,netral,1.00,Akun @kabarrnusantara melaporkan pernyataan Prabowo Subianto yang berencana mengadakan rapat terbatas. Rapat tersebut akan membahas langkah-langkah untuk mewujudkan pendidikan gratis di seluruh satuan pendidikan negeri.,"Sentimen netral masuk akal karena akun ini hanya menyampaikan informasi atau pernyataan dari Prabowo secara objektif, tanpa menambahkan opini, dukungan, atau kritik terhadap rencana tersebut.",
6,agenda_pertemuan,7,@vooxid_,0.31,netral,1.00,Akun @vooxid_ melapork